In [24]:
import os
import sys
os.chdir(r"C:\Users\juszynski\Desktop\wspace\coordinating-quantifiers")

In [25]:
import calculator
import importlib
from fractions import Fraction
from bokeh.plotting import figure
from bokeh.plotting import figure, show, output_notebook
import numpy as np

In [26]:
importlib.reload(calculator)

<module 'calculator' from 'C:\\Users\\juszynski\\Desktop\\wspace\\coordinating-quantifiers\\v2\\calculator.py'>

In [27]:
from calculator import QuotientCalculator, NumericCalculator

In [28]:
def support_and_pdf(p, density):
    '''Returns support and pdf for the given stimuli p'''
    return density.support(), density.pdf(p)

## Quotient with/without ANS

In [29]:
ans = True

if ans:
    quotient_stimuli_onfly, stimuli_density_onfly, quotient_calculator_onfly  = QuotientCalculator.from_description_with_ans(sigma_scalar=.1)
    quotient_stimuli, stimuli_density,quotient_calculator  = QuotientCalculator.load_from_file_with_ans()
else:
    quotient_stimuli_onfly, stimuli_density_onfly, quotient_calculator_onfly  = QuotientCalculator.from_description_with_no_ans()
    quotient_stimuli, stimuli_density, quotient_calculator  = QuotientCalculator.load_from_file_with_no_ans()

#assert quotient_stimuli == quotient_stimuli_onfly

del quotient_stimuli_onfly

In [30]:
from ipywidgets import widgets, HBox, VBox
from IPython.display import display, clear_output
style = {'description_width': 'initial'}

stimuli_selection_widget = widgets.SelectionSlider(
    options=quotient_stimuli,
    description='Stimuli range',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d',
    style=style
)
# .on_change('value', lambda x,y,z: print(value))
display(stimuli_selection_widget)


SelectionSlider(continuous_update=False, description='Stimuli range', options=(Fraction(1, 100), Fraction(1, 9…

In [32]:
p = Fraction(stimuli_selection_widget.value)

fig = figure(title='Pdf(X)', x_axis_label='Support', y_axis_label='Pdf(X)')

support, pdf = support_and_pdf(p, stimuli_density_onfly)
support1, pdf1 = support_and_pdf(p, stimuli_density)

zeros = np.where(pdf==0)[0]
zeros = [support[z] for z in zeros]
zeros1 = np.where(pdf1==0)[0]
zeros1 = [support1[z] for z in zeros1]

fig.line(support, pdf, line_width=1, legend_label='Recomputed on fly', color='blue')
fig.line(support1, pdf1, line_width=1, legend_label='From Franek file', color='red')
# fig.circle(float(p), -.05, legend_label='Stimuli', color='red', size=2)
fig.circle(zeros, .1, legend_label='Zeroed', size=1, color='blue')
fig.circle(zeros1, .2, legend_label='Zeroed1 franek', color='red',  size=1)

output_notebook()
show(fig)

Loading BokehJS ...

# Numeric with/without ANS

In [112]:
ans = True

if ans:
    numeric_stimuli_onfly, numeric_stimuli_density_onfly, numeric_calculator_onfly  = NumericCalculator.from_description_with_ans()
    numeric_stimuli, numeric_stimuli_density, numeric_calculator = NumericCalculator.load_from_file_with_ans()
else:
    numeric_stimuli_onfly, numeric_stimuli_density_onfly, numeric_calculator_onfly  = NumericCalculator.from_description_with_no_ans(sigma=1/3)
    numeric_stimuli, numeric_stimuli_density, numeric_calculator = NumericCalculator.load_from_file_with_no_ans()

#assert numeric_stimuli == numeric_stimuli_onfly

del numeric_stimuli_onfly

In [55]:
from ipywidgets import widgets, HBox, VBox
from IPython.display import display, clear_output
style = {'description_width': 'initial'}

stimuli_selection_widget = widgets.SelectionSlider(
    options=numeric_stimuli,
    description='Stimuli range',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d',
    style=style
)
# .on_change('value', lambda x,y,z: print(value))
display(stimuli_selection_widget)


SelectionSlider(continuous_update=False, description='Stimuli range', options=(1, 2, 3, 4, 5, 6, 7, 8, 9, 10, …

In [136]:
 np.sum(numeric_calculator.get_rxr()[2])

1.0738316315664134

In [138]:
mu1, sigma1 = params[1]
mu2, sigma2 = params[600]

ys = dot(mu1, mu2, sigma1, sigma2, np.array(stimuli_density.support()))
fig = figure(title='Pdf(X)', x_axis_label='Support', y_axis_label='Pdf(X)')
i = 15
norm = np.sum(numeric_calculator_onfly.get_rxr()[i])
fig.line(stimuli_density.support(), numeric_calculator.get_rxr()[i], line_width=1, legend_label=' franek', color='red')
fig.line(stimuli_density.support(), numeric_calculator_onfly.get_rxr()[i]/norm, line_width=1, legend_label=' on fly', color='blue')

# fig.line(support1, pdf1, line_width=1, legend_label='From Franek file', color='red')
output_notebook()
show(fig)

Loading BokehJS ...

In [113]:
p = int(stimuli_selection_widget.value)

fig = figure(title='Pdf(X)', x_axis_label='Support', y_axis_label='Pdf(X)')

support, pdf = support_and_pdf(p, numeric_stimuli_density_onfly)
support1, pdf1 = support_and_pdf(p, numeric_stimuli_density)

zeros = np.where(pdf==0)[0]
zeros = [support[z] for z in zeros]
zeros1 = np.where(pdf1==0)[0]
zeros1 = [support1[z] for z in zeros1]

fig.line(support, pdf, line_width=1, legend_label='Recomputed on fly', color='blue')
fig.line(support1, pdf1, line_width=1, legend_label='From Franek file', color='red')
fig.circle(zeros, .01, legend_label='Zeroed', size=1, color='blue')
fig.circle(zeros1, .02, legend_label='Zeroed1 franek', color='green',  size=1)

output_notebook()
show(fig)

KeyError: 0

In [56]:
def estimate_m_std(p):
    support, pdf_data = support_and_pdf(p, quotient_calculator)
    
    # Normalize the PDF data to ensure it integrates to 1
    pdf_data /= np.trapz(pdf_data, support)
    
    # Perform inverse transform sampling to generate random samples
    def inverse_transform_sampling(pdf_data, support, num_samples=1000):
        cdf = np.cumsum(pdf_data * (support[1] - support[0]))
        cdf /= cdf[-1]  # Normalize the CDF
    
        # Generate random samples from a uniform distribution
        random_samples = np.random.uniform(0, 1, num_samples)
    
        # Use the inverse CDF to map uniform samples to samples from the desired distribution
        inverse_samples = np.interp(random_samples, cdf, support)
    
        return inverse_samples
    
    # Generate random samples from the Gaussian distribution
    num_samples =1000  # Adjust as needed
    random_samples = inverse_transform_sampling(pdf_data, support, num_samples)

    return float(p),np.round(np.mean(random_samples), 4), np.round(np.std(random_samples), 4)

estimated_m_and_std = [estimate_m_std(p) for p in quotient_stimuli]

<function __main__.estimate_m_std(p)>

In [55]:
fig = figure(title='mu/std', x_axis_label='Mean', y_axis_label='Std')
xs = [x for x, _, _ in estimated_m_and_std]
ys = [y for _, _, y in estimated_m_and_std]

fig.line(xs, ys, line_width=1, legend_label='Recomputed on fly', color='blue')
output_notebook()
show(fig)

Loading BokehJS ...